# DeepVAR Land Cover Forecasting Pipeline (ILR-Compositional)

**What changed (from the original pipeline):**

1. **True multivariate joint modeling.** Targets are now the 6 **ILR coordinates** of the 7-class
   compositional vector (Aitchison's isometric log-ratio transform). The model learns a joint
   distribution over an unbounded $\mathbb{R}^6$ vector instead of 7 independent univariate
   Gaussians on $[0, 1]$. Predictions are inverted back to the simplex at the end.
2. **`batch_size` is now `multiple-of-6 × n_barangays_per_batch`** with `batch_sampler='synchronized'`,
   so all 6 ILR coords of a barangay co-occur in every batch — which is what makes the rank-`r`
   covariance in `MultivariateNormalDistributionLoss` meaningful.
3. **`target_normalizer` is now per-series `GroupNormalizer`** instead of `None`. ILR coords have
   wildly different scales across barangays (some near zero, some huge); normalizing per-series
   keeps the Gaussian likelihood well-conditioned.
4. **Quarter is now a `time_varying_known_categorical`.** Free seasonality signal previously thrown
   away.
5. **`limit_train_batches` removed from the final fit** (kept only for Optuna trials).
6. **Post-hoc simplex projection retained as belt-and-suspenders.** ILR inversion already produces
   valid simplex points, but we keep `clip→renormalize` after numerical inversion as a safety net
   against floating-point drift.

| Mode | Description |
|------|-------------|
| `'full'` | Tune + train + eval + predict |
| `'eval_predict'` | Load saved model → eval + predict |
| `'predict_only'` | Load saved model → predict only |


In [ ]:
# ============================================================
#  CELL 1 — CONFIG  (edit everything here)
# ============================================================

# ── Run mode ────────────────────────────────────────────────
RUN_MODE = 'full'   # 'full' | 'eval_predict' | 'predict_only'

# ── File paths ───────────────────────────────────────────────
DATA_PATH   = '/content/time-series-data.csv'
PTH_PATH    = '/content/deepvar_final.pth'
PARAMS_PATH = '/content/best_params.json'
DATASET_SCHEMA_PATH = '/content/training_schema.pkl'   # saved by Cell 7, loaded by Cell 8

# ── Sequence lengths ─────────────────────────────────────────
PREDICTION_LENGTH = 4
ENCODER_LENGTH    = 12

# ── Land cover columns (ORDER MATTERS — fixed for ILR basis) ─
LAND_COVER_COLS = [
    'bare_ground', 'built_area', 'crops', 'grass',
    'shrub_and_scrub', 'trees', 'water',
]
N_LC      = len(LAND_COVER_COLS)        # 7
N_ILR     = N_LC - 1                    # 6 ILR coordinates per barangay
ILR_COORD_NAMES = [f'ilr_{k}' for k in range(N_ILR)]

# Numerical floor used when log-transforming zeros (additive replacement).
# Land cover percentages frequently contain exact zeros, which log() cannot
# handle.  We replace with a small floor and renormalize before ILR.
COMPOSITIONAL_FLOOR = 1e-4

# ── Train / val / test split (CHRONOLOGICAL, NON-OVERLAPPING) ─
# Boundaries are mutually exclusive in time to prevent leakage:
#   Train : 2016 .. VAL_FROM_YEAR - 1   (model fits on this)
#   Val   : VAL_FROM_YEAR .. TRAIN_UNTIL_YEAR  (early stopping / hp tuning)
#   Test  : TEST_FROM_YEAR ..            (final unbiased eval)
VAL_FROM_YEAR    = 2022   # validation starts here
TRAIN_UNTIL_YEAR = 2023   # last year of validation (== last year of "seen" data)
TEST_FROM_YEAR   = 2024   # test starts here

assert VAL_FROM_YEAR <= TRAIN_UNTIL_YEAR, \
    f'VAL_FROM_YEAR ({VAL_FROM_YEAR}) must be <= TRAIN_UNTIL_YEAR ({TRAIN_UNTIL_YEAR})'
assert TRAIN_UNTIL_YEAR < TEST_FROM_YEAR, \
    f'TRAIN_UNTIL_YEAR ({TRAIN_UNTIL_YEAR}) must be < TEST_FROM_YEAR ({TEST_FROM_YEAR})'

# ── Future forecast target ───────────────────────────────────
TARGET_YEAR    = 2026
TARGET_QUARTER = 1

# ── Memory controls ──────────────────────────────────────────
N_SAMPLES       = 50
EVAL_BATCH_SIZE = 36   # multiple of N_ILR (6); see Cell 5 for why

# ── Optuna tuning (only used when RUN_MODE == 'full') ────────
N_TRIALS        = 30
TUNE_TIMEOUT_S  = 3600

# ── Internal state (do not edit) ─────────────────────────────
YEAR_ORIGIN = None

import os
print(f'RUN_MODE    : {RUN_MODE}')
print(f'DATA_PATH   : {DATA_PATH}   exists={os.path.isfile(DATA_PATH)}')
print(f'PTH_PATH    : {PTH_PATH}    exists={os.path.isfile(PTH_PATH)}')
print(f'PARAMS_PATH : {PARAMS_PATH} exists={os.path.isfile(PARAMS_PATH)}')
print(f'N_SAMPLES={N_SAMPLES}  EVAL_BATCH_SIZE={EVAL_BATCH_SIZE}  '
      f'(must be multiple of N_ILR={N_ILR})')
assert EVAL_BATCH_SIZE % N_ILR == 0, \
    f'EVAL_BATCH_SIZE ({EVAL_BATCH_SIZE}) must be a multiple of N_ILR ({N_ILR})'

assert RUN_MODE in ('full', 'eval_predict', 'predict_only'), \
    f"RUN_MODE must be 'full', 'eval_predict', or 'predict_only' — got '{RUN_MODE}'"

if RUN_MODE in ('eval_predict', 'predict_only'):
    for p, label in [(PTH_PATH, 'PTH_PATH'), (PARAMS_PATH, 'PARAMS_PATH'),
                     (DATASET_SCHEMA_PATH, 'DATASET_SCHEMA_PATH')]:
        if not os.path.isfile(p):
            raise FileNotFoundError(
                f'{label} not found: {p}\n'
                f"Either set RUN_MODE='full' to train from scratch, "
                f'or point {label} to your saved file.'
            )
print('Config OK.')


RUN_MODE    : full
DATA_PATH   : /content/time-series-data.csv   exists=True
PTH_PATH    : /content/deepvar_final.pth    exists=False
PARAMS_PATH : /content/best_params.json exists=False
N_SAMPLES=50  EVAL_BATCH_SIZE=36  (must be multiple of N_ILR=6)
Config OK.


In [ ]:
# ============================================================
#  CELL 2 — INSTALL DEPENDENCIES
# ============================================================
!pip install -q optuna optuna-integration[pytorch_lightning] \
              pytorch-forecasting lightning \
              pandas numpy scikit-learn statsmodels scipy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 59.0 MB/s eta 0:00:00


In [ ]:
# ============================================================
#  CELL 3 — IMPORTS & UTILITIES
# ============================================================
import gc, json, os, warnings
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import lightning.pytorch as pl

from scipy.stats import shapiro, probplot
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.stattools import durbin_watson

from pytorch_forecasting import DeepAR, TimeSeriesDataSet
from pytorch_forecasting.data import NaNLabelEncoder, GroupNormalizer
from pytorch_forecasting.metrics import MultivariateNormalDistributionLoss

warnings.filterwarnings('ignore')

def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('Imports OK.')


Imports OK.


In [ ]:
# ============================================================
#  CELL 4 — SHARED HELPER FUNCTIONS  (with ILR transform)
# ============================================================

# ── ILR (Isometric Log-Ratio) transform ──────────────────────
# Helmert / orthonormal contrast basis for the simplex.  Maps a D-part
# composition on the simplex to D-1 unbounded reals.  The basis is fixed
# given LAND_COVER_COLS order, so transform/inverse round-trips exactly
# (up to floating-point precision).
def _build_ilr_basis(D: int) -> np.ndarray:
    """Helmert sub-matrix: orthonormal basis for the (D-1)-dim subspace
    of R^D orthogonal to the all-ones vector. Shape (D-1, D)."""
    H = np.zeros((D - 1, D))
    for k in range(D - 1):
        coef = 1.0 / np.sqrt((k + 1) * (k + 2))
        H[k, : k + 1] =  coef
        H[k, k + 1]   = -(k + 1) * coef
    return H

ILR_BASIS = _build_ilr_basis(N_LC)   # shape (6, 7)

def compose_floor_and_close(x: np.ndarray, floor: float = COMPOSITIONAL_FLOOR) -> np.ndarray:
    """Replace exact zeros with `floor`, renormalize so rows sum to 1.
    `x` may be (..., N_LC). Standard pre-step before any log-ratio."""
    x = np.asarray(x, dtype=np.float64)
    x = np.where(x <= 0, floor, x)
    s = x.sum(axis=-1, keepdims=True)
    return x / np.where(s == 0, 1.0, s)

def ilr_transform(x: np.ndarray) -> np.ndarray:
    """Simplex composition (..., N_LC) → ILR coords (..., N_LC-1)."""
    x = compose_floor_and_close(x)
    log_x = np.log(x)
    return log_x @ ILR_BASIS.T   # (..., N_LC-1)

def ilr_inverse(z: np.ndarray) -> np.ndarray:
    """ILR coords (..., N_ILR) → simplex composition (..., N_LC).
    Numerically stable softmax-of-(B^T z)."""
    z = np.asarray(z, dtype=np.float64)
    log_x = z @ ILR_BASIS                       # (..., N_LC)
    log_x = log_x - log_x.max(axis=-1, keepdims=True)
    x = np.exp(log_x)
    return x / x.sum(axis=-1, keepdims=True)

# ── Self-test the round-trip on a random simplex sample ──────
_rng = np.random.default_rng(0)
_test = _rng.dirichlet(np.ones(N_LC), size=5)
_back = ilr_inverse(ilr_transform(_test))
assert np.allclose(_test, _back, atol=1e-10), 'ILR round-trip broken'
print('ILR transform: OK (round-trip max err ='
      f' {np.max(np.abs(_test - _back)):.2e})')

# ── Data prep ────────────────────────────────────────────────
def prep_frame(frame, year_origin):
    df = frame.sort_values(['baranggay_id', 'year', 'quarter']).reset_index(drop=True).copy()
    df['time_idx']      = (df['year'] - year_origin) * 4 + (df['quarter'] - 1)
    df['baranggay_id']  = df['baranggay_id'].astype(str)
    df[LAND_COVER_COLS] = df[LAND_COVER_COLS].astype(float)
    return df

def melt_to_ilr(frame: pd.DataFrame) -> pd.DataFrame:
    """Wide composition → long ILR. One row per (barangay, time_idx, ilr_k).
    Keeps `quarter` as a known categorical for the model."""
    comp = frame[LAND_COVER_COLS].to_numpy()
    ilr  = ilr_transform(comp)                         # (N, N_ILR)
    ilr_df = pd.DataFrame(ilr, columns=ILR_COORD_NAMES)
    base = frame[['baranggay_id', 'time_idx', 'year', 'quarter']].reset_index(drop=True)
    wide = pd.concat([base, ilr_df], axis=1)
    out  = wide.melt(
        id_vars=['baranggay_id', 'time_idx', 'year', 'quarter'],
        value_vars=ILR_COORD_NAMES,
        var_name='ilr_coord', value_name='value',
    )
    out['series']  = (out['baranggay_id'] + '_' + out['ilr_coord']).astype(str)
    out['quarter'] = out['quarter'].astype(str)        # categorical
    return out.sort_values(['series', 'time_idx']).reset_index(drop=True)

def split_series_id(sid: str) -> tuple[str, str]:
    """'<bgy>_ilr_<k>' → ('<bgy>', 'ilr_<k>').  Robust to underscores in bgy id."""
    for cn in ILR_COORD_NAMES:
        if sid.endswith('_' + cn):
            return sid[:-(len(cn) + 1)], cn
    raise ValueError(f'Cannot parse series id: {sid!r}')

# ── ILR-space samples → simplex samples (with safety projection) ─
def ilr_samples_to_simplex(samples_ilr_by_series, series_index, prediction_length,
                           apply_projection: bool = True):
    """Convert per-(barangay, ilr_k) sample arrays into per-(barangay, lc) sample arrays.

    Inputs
    ------
    samples_ilr_by_series : np.ndarray of shape (n_series, n_samples, prediction_length)
        Predicted samples for every (barangay × ILR-coord) series in canonical order.
    series_index : list[str]
        The corresponding series ids of length n_series.
    apply_projection : bool
        After ilr_inverse → simplex, additionally clip→renormalize as a numerical safety net.

    Returns
    -------
    samples_lc : np.ndarray of shape (n_series_lc, n_samples, prediction_length)
        Where n_series_lc = n_barangays × N_LC.  Series ids come back as
        '<bgy>_<land_cover_name>' so they're directly compatible with the rest of the pipeline.
    series_index_lc : list[str]
    """
    n_series, n_samples, pred_len = samples_ilr_by_series.shape
    assert pred_len == prediction_length

    # Group rows by barangay; collect (n_ilr × n_samples × pred_len) per barangay.
    by_bgy = OrderedDict()
    for i, sid in enumerate(series_index):
        bgy, coord = split_series_id(sid)
        slot = by_bgy.setdefault(bgy, [None] * N_ILR)
        k = ILR_COORD_NAMES.index(coord)
        slot[k] = samples_ilr_by_series[i]

    out_samples = []
    out_sids    = []
    for bgy, slots in by_bgy.items():
        if any(s is None for s in slots):
            raise RuntimeError(f'Barangay {bgy!r} is missing one of the ILR coords '
                               f'in the prediction batch — cannot invert ILR.')
        # Stack: (N_ILR, n_samples, pred_len) → (n_samples, pred_len, N_ILR)
        z = np.stack(slots, axis=0).transpose(1, 2, 0)
        # Invert ILR per (sample, time): produces (n_samples, pred_len, N_LC)
        comp = ilr_inverse(z)
        if apply_projection:
            comp = np.clip(comp, 0.0, 1.0)
            s = comp.sum(axis=-1, keepdims=True)
            comp = np.where(s > 1e-8, comp / s, np.full_like(comp, 1.0 / N_LC))
        # Reshape to (N_LC, n_samples, pred_len), one row per LC class for this bgy
        comp_per_lc = comp.transpose(2, 0, 1)
        for k, lc in enumerate(LAND_COVER_COLS):
            out_samples.append(comp_per_lc[k])
            out_sids.append(f'{bgy}_{lc}')
    return np.stack(out_samples, axis=0), out_sids

# ── Probabilistic metric helpers (unchanged from original) ───
def crps_sample(y_true, samples):
    S = samples.shape[1]
    term1    = np.mean(np.abs(samples - y_true[:, None]), axis=1)
    s_sorted = np.sort(samples, axis=1)
    weights  = 2 * np.arange(1, S + 1) - S - 1
    term2    = (s_sorted * weights[None, :]).sum(axis=1) / (S * S)
    return float(np.mean(term1 - term2))

def interval_score(y_true, samples, alpha=0.1):
    lo = np.percentile(samples, 100 * alpha / 2,     axis=1)
    hi = np.percentile(samples, 100 * (1 - alpha/2), axis=1)
    return float(np.mean((hi - lo)
                         + (2/alpha) * np.maximum(lo - y_true, 0)
                         + (2/alpha) * np.maximum(y_true - hi, 0)))

def coverage(y_true, samples, alpha=0.1):
    lo = np.percentile(samples, 100 * alpha / 2,     axis=1)
    hi = np.percentile(samples, 100 * (1 - alpha/2), axis=1)
    return float(np.mean((y_true >= lo) & (y_true <= hi)))

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return float(1 - ss_res / (ss_tot + 1e-12))

def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    return float(np.mean(np.abs(y_true - y_pred) / (denom + 1e-8)) * 100)

def mean_bias(y_true, y_pred):
    return float(np.mean(y_pred - y_true))

def kl_divergence_simplex(p, q, eps=1e-9):
    p = np.clip(p, eps, 1); q = np.clip(q, eps, 1)
    return np.sum(p * np.log(p / q), axis=-1)

def skill_score_mae(y_true, y_pred, y_baseline):
    return float(1 - np.mean(np.abs(y_true - y_pred))
                   / (np.mean(np.abs(y_true - y_baseline)) + 1e-12))

def _samp_flat(samples, n_series, pred_len):
    return samples.transpose(0, 2, 1).reshape(n_series * pred_len, N_SAMPLES)

print('Helper functions defined.')


ILR transform: OK (round-trip max err = 2.22e-16)
Helper functions defined.


In [ ]:
# ============================================================
#  CELL 5 — LOAD DATA & BUILD DATASETS  (ILR + quarter covariate)
# ============================================================
# Three mutually-exclusive chronological splits — NO leakage between them:
#   Train : year < VAL_FROM_YEAR
#   Val   : VAL_FROM_YEAR <= year <= TRAIN_UNTIL_YEAR
#   Test  : year >= TEST_FROM_YEAR
#
# Validation and test datasets each prepend the last ENCODER_LENGTH timesteps
# of the prior split as *encoder context only*. The model never trains on val
# or test rows; min_prediction_idx ensures decoder targets fall strictly inside
# the held-out window.
df_raw = pd.read_csv(DATA_PATH)

df_train_raw = df_raw[df_raw['year'] <  VAL_FROM_YEAR].copy()
df_val_raw   = df_raw[(df_raw['year'] >= VAL_FROM_YEAR) &
                      (df_raw['year'] <= TRAIN_UNTIL_YEAR)].copy()
df_test_raw  = df_raw[df_raw['year'] >= TEST_FROM_YEAR].copy()
del df_raw; free()

# YEAR_ORIGIN is anchored to the earliest training year so time_idx values stay
# stable across all three splits (essential for from_dataset to work).
YEAR_ORIGIN = int(df_train_raw['year'].min())
df_train = prep_frame(df_train_raw, YEAR_ORIGIN)
df_val   = prep_frame(df_val_raw,   YEAR_ORIGIN)
df_test  = prep_frame(df_test_raw,  YEAR_ORIGIN)
del df_train_raw, df_val_raw, df_test_raw; free()

# Long format in ILR space — 6 series per barangay (ilr_0 … ilr_5).
df_train_long = melt_to_ilr(df_train)
df_val_long   = melt_to_ilr(df_val)
df_test_long  = melt_to_ilr(df_test)
del df_train, df_val, df_test; free()

# Cutoffs. The training dataset spans the train period only; validation and
# test predictions begin one step after their respective context cutoffs.
train_last_tidx     = df_train_long['time_idx'].max()
val_last_tidx       = df_val_long['time_idx'].max() if len(df_val_long) else train_last_tidx
val_context_cutoff  = train_last_tidx          # last train timestep -> last context step before val
test_context_cutoff = val_last_tidx            # last val timestep   -> last context step before test

# Per-series GroupNormalizer: ILR coords have very different scales across barangays
# (some land covers are absent → very negative ILR values; some are dominant → highly
# positive).  Normalising per-series keeps the Gaussian likelihood well-conditioned.
# IMPORTANT: fitted on TRAIN only — never on val/test, otherwise normaliser stats
# leak future information.
target_normalizer = GroupNormalizer(groups=['series'], transformation=None)

# `series` encoder must see every series id that will appear in any split, so it
# can map test-time barangays to embedding rows. This is metadata, not target
# leakage — embedding *vocabulary* is allowed; embedding *values* are learned
# from training only.
all_series_ids = pd.concat(
    [df_train_long['series'], df_val_long['series'], df_test_long['series']]
).astype(str).unique()

training = TimeSeriesDataSet(
    df_train_long,
    time_idx               = 'time_idx',
    target                 = 'value',
    group_ids              = ['series'],
    static_categoricals    = ['series'],
    categorical_encoders   = {'series': NaNLabelEncoder().fit(pd.Series(all_series_ids))},
    time_varying_unknown_reals      = ['value'],
    time_varying_known_reals        = ['time_idx'],
    time_varying_known_categoricals = ['quarter'],
    max_encoder_length     = ENCODER_LENGTH,
    max_prediction_length  = PREDICTION_LENGTH,
    target_normalizer      = target_normalizer,
)

# ---- VALIDATION dataset ----
# Concatenate the last ENCODER_LENGTH train rows (encoder context) with the full
# val period. Predictions must start at val_context_cutoff + 1 (= first val step).
df_train_tail = df_train_long[df_train_long['time_idx'] > val_context_cutoff - ENCODER_LENGTH]
df_combined_for_val = pd.concat(
    [df_train_tail, df_val_long],
    ignore_index=True,
).sort_values(['series', 'time_idx']).reset_index(drop=True)
validation = TimeSeriesDataSet.from_dataset(
    training, df_combined_for_val, min_prediction_idx=val_context_cutoff + 1,
)
del df_combined_for_val, df_train_tail; free()

# ---- TEST dataset ----
# Concatenate the last ENCODER_LENGTH val rows (encoder context) with the full
# test period. Predictions must start at test_context_cutoff + 1.
df_val_tail = df_val_long[df_val_long['time_idx'] > test_context_cutoff - ENCODER_LENGTH]
df_combined_for_test = pd.concat(
    [df_val_tail, df_test_long],
    ignore_index=True,
).sort_values(['series', 'time_idx']).reset_index(drop=True)
test_dataset = TimeSeriesDataSet.from_dataset(
    training, df_combined_for_test, min_prediction_idx=test_context_cutoff + 1,
)
del df_combined_for_test, df_val_tail, df_test_long, df_val_long; free()

# ---- TRAIN_EVAL dataset (in-sample diagnostic) ----
# One prediction window per series at the end of the training period. Used for
# over/underfitting diagnosis: comparing train vs val metrics tells us whether
# the model memorized the training data (low train + high val = overfit) or
# failed to learn enough (high train + high val = underfit).
# predict=True selects the LAST prediction window per group from the data.
train_eval_dataset = TimeSeriesDataSet.from_dataset(
    training, df_train_long, predict=True, stop_randomization=True,
)

n_barangays = df_train_long['baranggay_id'].nunique()
print(f'YEAR_ORIGIN : {YEAR_ORIGIN}')
print(f'Barangays   : {n_barangays}   (×{N_ILR} ILR coords = {n_barangays * N_ILR} series)')
print(f'')
print(f'Split       | Years                       | time_idx range')
print(f'------------+-----------------------------+-----------------')
print(f'Train       | < {VAL_FROM_YEAR}                       | 0 .. {train_last_tidx}')
print(f'Validation  | {VAL_FROM_YEAR} .. {TRAIN_UNTIL_YEAR}                 | {val_context_cutoff + 1} .. {val_last_tidx}')
print(f'Test        | >= {TEST_FROM_YEAR}                      | {test_context_cutoff + 1} ..')
print(f'')
print(f'Training      dataset : {len(training)} samples (model fits on this)')
print(f'Train-eval    dataset : {len(train_eval_dataset)} samples (in-sample diagnostic)')
print(f'Validation    dataset : {len(validation)} samples')
print(f'Test          dataset : {len(test_dataset)} samples')
print(f'Target space: ILR coordinates (R^{N_ILR}), unbounded.')
print(f'Reminder    : batch_size must be a multiple of {N_ILR} so all coords '
      f'of a barangay co-occur in each batch (synchronized sampling).')


YEAR_ORIGIN : 2016
Barangays   : 129   (×6 ILR coords = 774 series)

Split       | Years                       | time_idx range
------------+-----------------------------+-----------------
Train       | < 2022                       | 0 .. 23
Validation  | 2022 .. 2023                 | 24 .. 31
Test        | >= 2024                      | 32 ..

Training      dataset : 6966 samples (model fits on this)
Train-eval    dataset : 774 samples (in-sample diagnostic)
Validation    dataset : 3870 samples
Test          dataset : 774 samples
Target space: ILR coordinates (R^6), unbounded.
Reminder    : batch_size must be a multiple of 6 so all coords of a barangay co-occur in each batch (synchronized sampling).


In [ ]:
# ============================================================
#  CELL 6 — OPTUNA TUNING  [full mode only]
# ============================================================
if RUN_MODE == 'full':
    import optuna
    from optuna.integration import PyTorchLightningPruningCallback

    # batch_size must be a multiple of N_ILR (=6) so every barangay's full
    # ILR vector is present in each batch — that's what makes the rank-r
    # MultivariateNormal covariance over batch dim meaningful.
    BATCH_CHOICES = [6 * 6, 6 * 12]   # 36, 72  (≈ 6 and 12 barangays per batch)

    def objective(trial: optuna.Trial) -> float:
        pl.seed_everything(42)
        hidden_size = trial.suggest_categorical('hidden_size', [32, 64, 128])
        rnn_layers  = trial.suggest_int('rnn_layers', 1, 2)
        dropout     = trial.suggest_float('dropout', 0.0, 0.2, step=0.05)
        lr          = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
        rank        = trial.suggest_categorical('rank', [2, 3, 6])
        batch_size  = trial.suggest_categorical('batch_size', BATCH_CHOICES)
        grad_clip   = trial.suggest_categorical('grad_clip', [0.05, 0.1, 0.5])

        train_dl = training.to_dataloader(
            train=True, batch_size=batch_size, num_workers=0, batch_sampler='synchronized',
        )
        val_dl = validation.to_dataloader(
            train=False, batch_size=batch_size, num_workers=0, batch_sampler='synchronized',
        )
        model = DeepAR.from_dataset(
            training,
            learning_rate=lr, hidden_size=hidden_size, rnn_layers=rnn_layers,
            dropout=dropout, optimizer='Adam',
            loss=MultivariateNormalDistributionLoss(rank=rank),
            log_interval=-1, log_val_interval=-1,
        )
        trainer = pl.Trainer(
            max_epochs=30,
            accelerator='gpu' if torch.cuda.is_available() else 'cpu',
            gradient_clip_val=grad_clip,
            limit_train_batches=50,
            enable_model_summary=False,
            enable_checkpointing=False,
            logger=False,
            callbacks=[PyTorchLightningPruningCallback(trial, monitor='val_loss')],
        )
        trainer.fit(model, train_dataloaders=train_dl, val_dataloaders=val_dl)
        return trainer.callback_metrics.get('val_loss', float('inf')).item()

    study = optuna.create_study(
        direction='minimize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
        sampler=optuna.samplers.TPESampler(seed=42),
        study_name='deepvar_landcover_ilr',
    )
    study.optimize(objective, n_trials=N_TRIALS, timeout=TUNE_TIMEOUT_S, show_progress_bar=True)

    print(f'\nBest val_loss : {study.best_value:.6f}')
    print(f'Best params   : {study.best_params}')

    with open(PARAMS_PATH, 'w') as f:
        json.dump(study.best_params, f, indent=4)
    print(f'Saved → {PARAMS_PATH}')

    import optuna.visualization.matplotlib as optuna_mpl
    for plot_fn in [optuna_mpl.plot_optimization_history,
                    optuna_mpl.plot_param_importances,
                    optuna_mpl.plot_parallel_coordinate]:
        try:
            ax = plot_fn(study)
            plt.tight_layout(); plt.show(); plt.close()
        except Exception:
            pass
else:
    print(f'[SKIPPED] Tuning — RUN_MODE={RUN_MODE}')


[I 2026-05-02 13:25:23,766] A new study created in memory with name: deepvar_landcover_ilr


  0%|          | 0/30 [00:00<?, ?it/s]

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

INFO: `Trainer.fit` stopped: `max_epochs=30` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=30` reached.


...

[I 2026-05-02 14:23:12,534] Trial 17 finished with value: 3484869.75 and parameters: {'hidden_size': 64, 'rnn_layers': 1, 'dropout': 0.1, 'lr': 0.00013096503299659426, 'rank': 3, 'batch_size': 36, 'grad_clip': 0.5}. Best is trial 14 with value: 1493990.25.


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

INFO: `Trainer.fit` stopped: `max_epochs=30` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=30` reached.


[I 2026-05-02 14:26:28,348] Trial 18 finished with value: 58091420.0 and parameters: {'hidden_size': 32, 'rnn_layers': 1, 'dropout': 0.2, 'lr': 0.000244754598579926, 'rank': 3, 'batch_size': 36, 'grad_clip': 0.05}. Best is trial 14 with value: 1493990.25.

Best val_loss : 1493990.250000
Best params   : {'hidden_size': 64, 'rnn_layers': 1, 'dropout': 0.15000000000000002, 'lr': 0.00010055452122808927, 'rank': 3, 'batch_size': 36, 'grad_clip': 0.5}
Saved → /content/best_params.json


In [ ]:
# ============================================================
#  CELL 7 — FINAL TRAINING  [full mode only]
# ============================================================
if RUN_MODE == 'full':
    from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

    with open(PARAMS_PATH) as f:
        best = json.load(f)
    print('Training with params:', best)

    train_dataloader = training.to_dataloader(
        train=True,  batch_size=best['batch_size'], num_workers=0, batch_sampler='synchronized',
    )
    val_dataloader = validation.to_dataloader(
        train=False, batch_size=best['batch_size'], num_workers=0, batch_sampler='synchronized',
    )

    net = DeepAR.from_dataset(
        training,
        learning_rate=best['lr'],
        hidden_size=best['hidden_size'],
        rnn_layers=best['rnn_layers'],
        dropout=best['dropout'],
        optimizer='Adam',
        loss=MultivariateNormalDistributionLoss(rank=best['rank']),
        log_interval=10, log_val_interval=1,
    )

    trainer = pl.Trainer(
        max_epochs=100,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        gradient_clip_val=best['grad_clip'],
        # NOTE: limit_train_batches REMOVED — final fit sees the full training set.
        enable_model_summary=True,
        callbacks=[
            EarlyStopping(monitor='val_loss', min_delta=1e-4, patience=10, verbose=True, mode='min'),
            ModelCheckpoint(monitor='val_loss',
                            filename='deepvar-best-{epoch:02d}-{val_loss:.4f}',
                            save_top_k=1, mode='min'),
        ],
    )
    trainer.fit(net, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

    best_model = DeepAR.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)
    torch.save(best_model.state_dict(), PTH_PATH)
    print(f'Model saved → {PTH_PATH}')

    # Save the TimeSeriesDataSet schema so eval/predict modes can rebuild new
    # datasets via from_dataset() with identical encoders, feature ordering,
    # and target normalization.  Without this, fresh TimeSeriesDataSet() calls
    # in Cells 13 and 15 produce a slightly different feature width and the
    # model raises 'mat1 and mat2 shapes cannot be multiplied'.
    training.save(DATASET_SCHEMA_PATH)
    print(f'Training schema saved → {DATASET_SCHEMA_PATH}')
else:
    print(f'[SKIPPED] Training — RUN_MODE={RUN_MODE}')


Training with params: {'hidden_size': 64, 'rnn_layers': 1, 'dropout': 0.15000000000000002, 'lr': 0.00010055452122808927, 'rank': 3, 'batch_size': 36, 'grad_clip': 0.5}


INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name                   ┃ Type                               ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss                   │ MultivariateNormalDistributionLoss │      0 │ train │     0 │
│ 1 │ logging_metrics        │ ModuleList                         │      0 │ train │     0 │
│ 2 │ embeddings             │ MultiEmbedding                     │ 51.1 K │ train │     0 │
│ 3 │ rnn                    │ LSTM                               │ 35.1 K │ train │     0 │
│ 4 │ distribution_projector │ Linear                             │    325 │ train │     0 │
└───┴────────────────────────┴────────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 86.5 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 86.5 K                                                                                               
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 13                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

INFO: Metric val_loss improved. New best score: 108373.070
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved. New best score: 108373.070
INFO: Monitored metric val_loss did not improve in the last 10 records. Best score: 108373.070. Signaling Trainer to stop.
INFO:lightning.pytorch.callbacks.early_stopping:Monitored metric val_loss did not improve in the last 10 records. Best score: 108373.070. Signaling Trainer to stop.


Model saved → /content/deepvar_final.pth
Training schema saved → /content/training_schema.pkl


In [ ]:
# ============================================================
#  CELL 8 — LOAD SAVED MODEL  [eval_predict / predict_only]
# ============================================================
if RUN_MODE in ('eval_predict', 'predict_only'):
    with open(PARAMS_PATH) as f:
        params = json.load(f)

    required = {'lr', 'hidden_size', 'rnn_layers', 'dropout', 'rank'}
    missing  = required - params.keys()
    if missing:
        raise KeyError(f'best_params.json missing keys: {missing}')
    print('Hyperparameters:', params)

    # Load the exact training schema (encoders, feature ordering, normalizer).
    # We need this not just to rebuild the model but also for from_dataset()
    # calls in Cells 13 and 15.
    training = TimeSeriesDataSet.load(DATASET_SCHEMA_PATH)
    print(f'Training schema loaded ← {DATASET_SCHEMA_PATH}')

    # Rebuild a fresh validation dataset placeholder (Cell 5's `validation`
    # variable is not in scope in eval_predict / predict_only modes).  In
    # eval_predict mode Cell 9 needs a real validation/test dataset; we read
    # the CSV here and rebuild them via from_dataset().
    if RUN_MODE == 'eval_predict':
        # Rebuild val/test using the SAME chronological, non-overlapping logic
        # as Cell 5. The training dataset itself was reloaded from the saved
        # schema above, so we only need val + test here.
        df_eval_raw = pd.read_csv(DATA_PATH)
        if YEAR_ORIGIN is None:
            YEAR_ORIGIN = int(df_eval_raw['year'].min())

        df_eval_train_raw = df_eval_raw[df_eval_raw['year'] <  VAL_FROM_YEAR].copy()
        df_eval_val_raw   = df_eval_raw[(df_eval_raw['year'] >= VAL_FROM_YEAR) &
                                        (df_eval_raw['year'] <= TRAIN_UNTIL_YEAR)].copy()
        df_eval_test_raw  = df_eval_raw[df_eval_raw['year'] >= TEST_FROM_YEAR].copy()
        del df_eval_raw

        df_eval_train = melt_to_ilr(prep_frame(df_eval_train_raw, YEAR_ORIGIN))
        df_eval_val   = melt_to_ilr(prep_frame(df_eval_val_raw,   YEAR_ORIGIN))
        df_eval_test  = melt_to_ilr(prep_frame(df_eval_test_raw,  YEAR_ORIGIN))
        del df_eval_train_raw, df_eval_val_raw, df_eval_test_raw

        train_last_tidx_e   = df_eval_train['time_idx'].max()
        val_last_tidx_e     = df_eval_val['time_idx'].max() if len(df_eval_val) else train_last_tidx_e
        val_context_cutoff_e  = train_last_tidx_e
        test_context_cutoff_e = val_last_tidx_e

        # Validation: train tail (encoder context) + val period
        df_train_tail_e = df_eval_train[df_eval_train['time_idx'] > val_context_cutoff_e - ENCODER_LENGTH]
        df_combined_for_val = pd.concat(
            [df_train_tail_e, df_eval_val],
            ignore_index=True,
        ).sort_values(['series', 'time_idx']).reset_index(drop=True)
        validation = TimeSeriesDataSet.from_dataset(
            training, df_combined_for_val, min_prediction_idx=val_context_cutoff_e + 1,
        )
        del df_combined_for_val, df_train_tail_e

        # Test: val tail (encoder context) + test period
        df_val_tail_e = df_eval_val[df_eval_val['time_idx'] > test_context_cutoff_e - ENCODER_LENGTH]
        df_combined_for_test = pd.concat(
            [df_val_tail_e, df_eval_test],
            ignore_index=True,
        ).sort_values(['series', 'time_idx']).reset_index(drop=True)
        test_dataset = TimeSeriesDataSet.from_dataset(
            training, df_combined_for_test, min_prediction_idx=test_context_cutoff_e + 1,
        )

        # Train-eval: in-sample diagnostic (last prediction window per series, within train period)
        train_eval_dataset = TimeSeriesDataSet.from_dataset(
            training, df_eval_train, predict=True, stop_randomization=True,
        )

        del df_combined_for_test, df_val_tail_e, df_eval_test, df_eval_val, df_eval_train
        free()
        print(f'Rebuilt train_eval ({len(train_eval_dataset)}), '
              f'validation ({len(validation)}), test ({len(test_dataset)}) datasets.')

    # Build the model from the loaded schema.
    best_model = DeepAR.from_dataset(
        training,
        learning_rate=params['lr'],
        hidden_size=params['hidden_size'],
        rnn_layers=params['rnn_layers'],
        dropout=params['dropout'],
        optimizer='Adam',
        loss=MultivariateNormalDistributionLoss(rank=params['rank']),
    )

    state_dict = torch.load(PTH_PATH, map_location='cpu')
    built_shape = dict(best_model.named_parameters())['embeddings.embeddings.series.weight'].shape
    saved_shape = state_dict['embeddings.embeddings.series.weight'].shape
    assert built_shape == saved_shape, \
        f'Embedding shape mismatch: built {built_shape} vs saved {saved_shape}.\n' \
        'This typically means DATASET_SCHEMA_PATH does not correspond to PTH_PATH — ' \
        'retrain or point both paths to a matched (schema, weights) pair.'

    best_model.load_state_dict(state_dict, strict=True)
    del state_dict; free()
    best_model.eval()
    print(f'Model loaded from {PTH_PATH}')
    print(f'Parameters: {sum(p.numel() for p in best_model.parameters()):,}')
else:
    best_model.eval()
    print('Using freshly trained model.')


Using freshly trained model.


In [ ]:
# ============================================================
#  CELL 9 — EVAL DATALOADERS  [full / eval_predict]
# ============================================================
if RUN_MODE in ('full', 'eval_predict'):
    train_eval_dataloader = train_eval_dataset.to_dataloader(
        train=False, batch_size=EVAL_BATCH_SIZE, num_workers=0, batch_sampler='synchronized',
    )
    val_dataloader = validation.to_dataloader(
        train=False, batch_size=EVAL_BATCH_SIZE, num_workers=0, batch_sampler='synchronized',
    )
    test_dataloader = test_dataset.to_dataloader(
        train=False, batch_size=EVAL_BATCH_SIZE, num_workers=0, batch_sampler='synchronized',
    )
    print('Eval dataloaders ready (train_eval / val / test).')
else:
    print(f'[SKIPPED] Eval dataloaders — RUN_MODE={RUN_MODE}')


Eval dataloaders ready (train_eval / val / test).


In [ ]:
# ============================================================
#  CELL 10 — PREDICT (ILR space) → INVERT TO SIMPLEX
# ============================================================
# The model emits samples in ILR space. We:
#   1. Get raw ILR samples per series (shape: n_series_ilr × n_samples × pred_len)
#   2. Group by barangay, stack the 6 ILR coords, invert to simplex (N_LC fractions)
#   3. Apply belt-and-suspenders clip→renormalize for floating-point safety
#   4. Also re-derive the *ground-truth simplex values* by inverting the ILR ground truth
#      (so y_true and y_pred live in the same simplex space).
if RUN_MODE in ('full', 'eval_predict'):

    def _to_canonical(arr, n_samples):
        # pytorch-forecasting may return (B, T, S) or (B, S, T) depending on version.
        return arr.transpose(0, 2, 1) if arr.shape[2] == n_samples else arr

    def _ilr_truth_to_simplex(y_true_ilr, series_index):
        """Group y_true_ilr (n_series_ilr, pred_len) by barangay and invert ILR.
        Returns y_true_simplex shape (n_series_lc, pred_len) and the corresponding
        list of '<bgy>_<lc>' series ids."""
        pred_len = y_true_ilr.shape[1]
        by_bgy = OrderedDict()
        for i, sid in enumerate(series_index):
            bgy, coord = split_series_id(sid)
            slot = by_bgy.setdefault(bgy, [None] * N_ILR)
            k = ILR_COORD_NAMES.index(coord)
            slot[k] = y_true_ilr[i]
        out, sids = [], []
        for bgy, slots in by_bgy.items():
            if any(s is None for s in slots):
                raise RuntimeError(f'Missing ILR coord for barangay {bgy!r}')
            z = np.stack(slots, axis=0).transpose(1, 0)        # (pred_len, N_ILR)
            comp = ilr_inverse(z)                              # (pred_len, N_LC)
            for k, lc in enumerate(LAND_COVER_COLS):
                out.append(comp[:, k])
                sids.append(f'{bgy}_{lc}')
        return np.stack(out, axis=0), sids

    def run_predictions(dataloader, dataset, label):
        print(f'  Predicting {label} (n_samples={N_SAMPLES}) ...')
        with torch.no_grad():
            preds = best_model.predict(
                dataloader, mode='samples', n_samples=N_SAMPLES,
                trainer_kwargs=dict(accelerator='cpu'),
                return_y=True, return_x=True,
            )
        series_idx_ilr = list(dataset.x_to_index(preds.x)['series'])
        y_true_ilr     = preds.y[0].numpy().copy()             # (n_series_ilr, pred_len)
        raw            = preds.output.numpy().copy()
        del preds; free()

        canon = _to_canonical(raw, N_SAMPLES)                  # (n_series_ilr, n_samples, pred_len)
        del raw; free()

        # Invert ILR → simplex for both samples and ground truth.
        samples_simplex, series_idx_lc = ilr_samples_to_simplex(
            canon, series_idx_ilr, PREDICTION_LENGTH, apply_projection=True,
        )
        del canon; free()
        y_true_simplex, sids_check = _ilr_truth_to_simplex(y_true_ilr, series_idx_ilr)
        assert series_idx_lc == sids_check, 'series id ordering mismatch between samples and truth'

        pred_mean = samples_simplex.mean(axis=1).astype(np.float32)
        return dict(series_idx=series_idx_lc,
                    y_true=y_true_simplex.astype(np.float32),
                    pred_mean=pred_mean,
                    samples=samples_simplex.astype(np.float32))

    train_res = run_predictions(train_eval_dataloader, train_eval_dataset, 'train (in-sample)')
    val_res   = run_predictions(val_dataloader,         validation,         'validation')
    test_res  = run_predictions(test_dataloader,        test_dataset,       'test')

    train_si        = train_res['series_idx']
    train_y_true_s  = train_res['y_true']
    train_pred_mean = train_res['pred_mean']
    train_samples   = train_res['samples']

    val_si        = val_res['series_idx']
    val_y_true_s  = val_res['y_true']
    val_pred_mean = val_res['pred_mean']
    val_samples   = val_res['samples']

    test_si        = test_res['series_idx']
    test_y_true_s  = test_res['y_true']
    test_pred_mean = test_res['pred_mean']
    test_samples   = test_res['samples']

    print(f'train_samples: {train_samples.shape}  (n_series_lc, n_samples, pred_len)')
    print(f'val_samples  : {val_samples.shape}')
    print(f'test_samples : {test_samples.shape}')
else:
    print(f'[SKIPPED] Eval predictions — RUN_MODE={RUN_MODE}')


INFO: GPU available: True (cuda), used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytor

  Predicting train (in-sample) (n_samples=50) ...


INFO: GPU available: True (cuda), used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytor

  Predicting validation (n_samples=50) ...


INFO: GPU available: True (cuda), used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytor

  Predicting test (n_samples=50) ...
train_samples: (903, 50, 4)  (n_series_lc, n_samples, pred_len)
val_samples  : (903, 50, 4)
test_samples : (903, 50, 4)


In [ ]:
# ============================================================
#  CELL 11 — EXTENDED METRICS  [full / eval_predict]
# ============================================================
if RUN_MODE in ('full', 'eval_predict'):

    def compute_extended_metrics(y_true, pred_mean, samples, split_name):
        n_series, pred_len = y_true.shape
        yt  = y_true.flatten()
        yp  = pred_mean.flatten()
        smp = _samp_flat(samples, n_series, pred_len)

        mae_v  = mean_absolute_error(yt, yp)
        rmse_v = float(np.sqrt(mean_squared_error(yt, yp)))
        mape_v = float(np.mean(np.abs((yt - yp) / (yt + 1e-8))) * 100)
        sm_v   = smape(yt, yp)
        bias_v = mean_bias(yt, yp)
        r2_v   = r2_score(yt, yp)
        crps_v = crps_sample(yt, smp)
        is90_v = interval_score(yt, smp, alpha=0.10)
        c90_v  = coverage(yt, smp, alpha=0.10)
        is50_v = interval_score(yt, smp, alpha=0.50)
        c50_v  = coverage(yt, smp, alpha=0.50)
        ss_v   = skill_score_mae(yt, yp, np.repeat(y_true[:, :1], pred_len, axis=1).flatten())
        del smp; free()

        print(f'\n{"="*55}')
        print(f'  {split_name}')
        print(f'{"="*55}')
        print(f'  MAE      : {mae_v:.6f}')
        print(f'  RMSE     : {rmse_v:.6f}')
        print(f'  MAPE     : {mape_v:.2f}%')
        print(f'  sMAPE    : {sm_v:.2f}%')
        print(f'  Bias     : {bias_v:+.6f}  (+ = over-predict)')
        print(f'  R^2      : {r2_v:.4f}')
        print(f'  CRPS     : {crps_v:.6f}')
        print(f'  IS  90%  : {is90_v:.6f}  |  Cov 90%: {c90_v*100:.1f}%')
        print(f'  IS  50%  : {is50_v:.6f}  |  Cov 50%: {c50_v*100:.1f}%')
        print(f'  Skill    : {ss_v:+.4f}  (>0 beats persistence)')
        return dict(split=split_name, mae=mae_v, rmse=rmse_v, mape=mape_v, smape=sm_v,
                    bias=bias_v, r2=r2_v, crps=crps_v,
                    is90=is90_v, cov90=c90_v, is50=is50_v, cov50=c50_v, skill=ss_v)

    train_ext = compute_extended_metrics(train_y_true_s, train_pred_mean, train_samples,
                                         f'Train (in-sample, < {VAL_FROM_YEAR})')
    val_ext   = compute_extended_metrics(val_y_true_s,   val_pred_mean,   val_samples,
                                         f'Validation ({VAL_FROM_YEAR}-{TRAIN_UNTIL_YEAR}, held out for tuning)')
    test_ext  = compute_extended_metrics(test_y_true_s,  test_pred_mean,  test_samples,
                                         f'Test ({TEST_FROM_YEAR}+, fully unseen)')

    summary = pd.DataFrame([train_ext, val_ext, test_ext]).set_index('split')
    summary.columns = ['MAE','RMSE','MAPE%','sMAPE%','Bias','R^2',
                       'CRPS','IS_90','Cov_90%','IS_50','Cov_50%','SkillScore']
    print('\n' + '='*80)
    print('  FULL EVALUATION SUMMARY — Train vs Validation vs Test')
    print('='*80)
    print(summary.T.round(4).to_string())

    # ---- Over/underfitting diagnostic ----
    train_mae = train_ext['mae']
    val_mae   = val_ext['mae']
    test_mae  = test_ext['mae']
    gap_train_val = val_mae  - train_mae
    gap_val_test  = test_mae - val_mae
    print(f'\n  Diagnostic gaps (MAE):')
    print(f'    val  - train : {gap_train_val:+.6f}   (large positive = overfitting to train)')
    print(f'    test - val   : {gap_val_test:+.6f}   (large positive = distribution shift or val over-tuning)')
    if train_mae > 0.05 and val_mae > 0.05:
        print('    Verdict: high error on both train and val — likely UNDERFITTING.')
    elif gap_train_val > max(0.5 * train_mae, 0.02):
        print('    Verdict: val error >> train error — likely OVERFITTING.')
    elif gap_val_test > max(0.5 * val_mae, 0.02):
        print('    Verdict: test error >> val error — distribution shift in 2024+ or val over-tuned.')
    else:
        print('    Verdict: train/val/test errors are reasonably balanced — healthy generalization.')
else:
    print(f'[SKIPPED] Eval metrics — RUN_MODE={RUN_MODE}')



  Train (in-sample, < 2022)
  MAE      : 0.006236
  RMSE     : 0.019547
  MAPE     : 381.99%
  sMAPE    : 20.97%
  Bias     : -0.000000  (+ = over-predict)
  R^2      : 0.9963
  CRPS     : 0.004881
  IS  90%  : 0.051803  |  Cov 90%: 97.6%
  IS  50%  : 0.023135  |  Cov 50%: 88.3%
  Skill    : -0.2638  (>0 beats persistence)

  Validation (2022-2023, held out for tuning)
  MAE      : 0.006284
  RMSE     : 0.020507
  MAPE     : 489.98%
  sMAPE    : 22.16%
  Bias     : -0.000000  (+ = over-predict)
  R^2      : 0.9959
  CRPS     : 0.005066
  IS  90%  : 0.055749  |  Cov 90%: 96.6%
  IS  50%  : 0.024144  |  Cov 50%: 88.4%
  Skill    : -0.8166  (>0 beats persistence)

  Test (2024+, fully unseen)
  MAE      : 0.007458
  RMSE     : 0.025180
  MAPE     : 1011.51%
  sMAPE    : 22.32%
  Bias     : -0.000000  (+ = over-predict)
  R^2      : 0.9939
  CRPS     : 0.005297
  IS  90%  : 0.052753  |  Cov 90%: 97.0%
  IS  50%  : 0.024102  |  Cov 50%: 88.0%
  Skill    : -0.4872  (>0 beats persistence)

 

In [ ]:
# ============================================================
#  CELL 12 — PER-LAND-COVER METRICS & VISUALISATIONS  [full / eval_predict]
# ============================================================
def _parse_bgy_lc(sid: str) -> tuple[str, str]:
    """'<bgy>_<lc>' → ('<bgy>', '<lc>').  Robust to underscores in LC names."""
    for lc in LAND_COVER_COLS:
        if sid.endswith('_' + lc):
            return sid[:-(len(lc) + 1)], lc
    raise ValueError(f'Cannot parse simplex series id: {sid!r}')


def _per_lc_metrics(si, y_true_s, pred_mean, samples, split_label: str):
    """Compute per-land-cover metrics and render bar charts for one split."""
    lc_arr = np.array([_parse_bgy_lc(s)[1] for s in si])

    lc_rows = []
    for lc in LAND_COVER_COLS:
        mask  = lc_arr == lc
        if not mask.any(): continue
        yt    = y_true_s[mask].flatten()
        yp    = pred_mean[mask].flatten()
        n_sub = mask.sum()
        smp   = samples[mask].transpose(0, 2, 1).reshape(n_sub * PREDICTION_LENGTH, N_SAMPLES)
        lc_rows.append(dict(
            land_cover=lc,
            CRPS=crps_sample(yt, smp), IS_90=interval_score(yt, smp, alpha=0.10),
            Cov_90=coverage(yt, smp, alpha=0.10)*100,
            IS_50=interval_score(yt, smp, alpha=0.50), Cov_50=coverage(yt, smp, alpha=0.50)*100,
            Bias=mean_bias(yt, yp), R2=r2_score(yt, yp), sMAPE=smape(yt, yp),
            MAE=mean_absolute_error(yt, yp),
        ))
        del smp; free()

    lc_df = pd.DataFrame(lc_rows).set_index('land_cover')
    print(f'{split_label.upper()} SET — Per Land-Cover Metrics')
    print(lc_df.round(4).to_string())

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    lc_df['CRPS'].sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
    axes[0].set_title('CRPS by Land Cover (lower=better)')
    lc_df['IS_90'].sort_values().plot(kind='barh', ax=axes[1], color='darkorange')
    axes[1].set_title('Interval Score 90% PI (lower=better)')
    lc_df['Cov_90'].sort_values().plot(kind='barh', ax=axes[2], color='seagreen')
    axes[2].axvline(90, color='red', linestyle='--', label='Nominal 90%')
    axes[2].set_title('90% Coverage'); axes[2].legend(fontsize=8)
    plt.suptitle(f'Probabilistic Forecast Quality — {split_label} Set', fontsize=13)
    plt.tight_layout(); plt.show(); plt.close(); free()

    overall_mae = mean_absolute_error(y_true_s.flatten(), pred_mean.flatten())
    fig, ax = plt.subplots(figsize=(9, 4))
    lc_df['MAE'].sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.axvline(overall_mae, color='red', linestyle='--', label=f'Overall MAE={overall_mae:.4f}')
    ax.set_title(f'MAE by Land Cover — {split_label} Set'); ax.legend()
    plt.tight_layout(); plt.show(); plt.close(); free()

    return lc_df


if RUN_MODE in ('full', 'eval_predict'):
    train_lc_df = _per_lc_metrics(train_si, train_y_true_s, train_pred_mean, train_samples, 'Train (in-sample)')
    val_lc_df   = _per_lc_metrics(val_si,   val_y_true_s,   val_pred_mean,   val_samples,   'Validation')
    test_lc_df  = _per_lc_metrics(test_si,  test_y_true_s,  test_pred_mean,  test_samples,  'Test')

    # ---- Per-land-cover comparison: how does each LC class generalize? ----
    cmp = pd.concat({
        'Train_MAE':  train_lc_df['MAE'],
        'Val_MAE':    val_lc_df['MAE'],
        'Test_MAE':   test_lc_df['MAE'],
        'Train_CRPS': train_lc_df['CRPS'],
        'Val_CRPS':   val_lc_df['CRPS'],
        'Test_CRPS':  test_lc_df['CRPS'],
    }, axis=1)
    cmp['Val_minus_Train_MAE']  = cmp['Val_MAE']  - cmp['Train_MAE']
    cmp['Test_minus_Val_MAE']   = cmp['Test_MAE'] - cmp['Val_MAE']
    print('\n' + '='*80)
    print('  PER-LAND-COVER COMPARISON — Train vs Val vs Test')
    print('='*80)
    print(cmp.round(4).to_string())
    print('\n  Read: large Val-Train gap on a class = that class overfits.')
    print('        large Test-Val gap on a class = that class shifts post-2024.')
else:
    print(f'[SKIPPED] Per-LC metrics — RUN_MODE={RUN_MODE}')

TRAIN (IN-SAMPLE) SET — Per Land-Cover Metrics
                   CRPS   IS_90   Cov_90   IS_50   Cov_50    Bias      R2    sMAPE     MAE
land_cover                                                                                
bare_ground      0.0003  0.0040  98.6434  0.0012  94.5736 -0.0001 -0.0035  17.3254  0.0004
built_area       0.0162  0.1692  96.7054  0.0774  81.3953 -0.0105  0.9712   3.2829  0.0198
crops            0.0002  0.0018  97.6744  0.0007  90.1163  0.0002 -0.2372  20.1732  0.0003
grass            0.0013  0.0162  96.7054  0.0055  86.6279  0.0012  0.8666  25.2953  0.0019
shrub_and_scrub  0.0005  0.0073  97.6744  0.0021  86.0465  0.0012 -0.1746  36.4748  0.0014
trees            0.0156  0.1622  96.3178  0.0744  81.7829  0.0079  0.9692  32.1818  0.0196
water            0.0001  0.0020  99.2248  0.0005  97.2868  0.0002  0.3887  12.0695  0.0002
VALIDATION SET — Per Land-Cover Metrics
                   CRPS   IS_90   Cov_90   IS_50   Cov_50    Bias      R2    sMAPE     MAE
lan

In [ ]:
# ============================================================
#  CELL 13 — ROLLING FUTURE FORECAST  (all modes)
# ============================================================

def _parse_bgy_lc_inline(sid):
    """'<bgy>_<lc>' -> ('<bgy>', '<lc>'). Defined locally so this cell works
    even when Cell 12 is skipped (predict_only mode)."""
    for lc in LAND_COVER_COLS:
        if sid.endswith('_' + lc):
            return sid[:-(len(lc) + 1)], lc
    raise ValueError(f'Cannot parse simplex series id: {sid!r}')

df_all = pd.read_csv(DATA_PATH)
if YEAR_ORIGIN is None:
    YEAR_ORIGIN = int(df_all['year'].min())

def _prep(frame):
    df = frame.sort_values(['baranggay_id', 'year', 'quarter']).reset_index(drop=True).copy()
    df['time_idx']      = (df['year'] - YEAR_ORIGIN) * 4 + (df['quarter'] - 1)
    df['baranggay_id']  = df['baranggay_id'].astype(str)
    df[LAND_COVER_COLS] = df[LAND_COVER_COLS].astype(float)
    return df

df_context_ilr = melt_to_ilr(_prep(df_all))
del df_all; free()

last_obs_tidx  = int(df_context_ilr['time_idx'].max())
last_obs_year  = last_obs_tidx // 4 + YEAR_ORIGIN
last_obs_qtr   = last_obs_tidx %  4 + 1

target_tidx    = (TARGET_YEAR - YEAR_ORIGIN) * 4 + (TARGET_QUARTER - 1)
n_future_steps = target_tidx - last_obs_tidx

print(f'Last observed : {last_obs_year} Q{last_obs_qtr}  (time_idx={last_obs_tidx})')
print(f'Target        : {TARGET_YEAR} Q{TARGET_QUARTER}  (time_idx={target_tidx})')
print(f'Steps to roll : {n_future_steps}')

if n_future_steps <= 0:
    raise ValueError(
        f'Target {TARGET_YEAR} Q{TARGET_QUARTER} is not in the future. '
        f'Last observed is {last_obs_year} Q{last_obs_qtr}. '
        'Increase TARGET_YEAR / TARGET_QUARTER in Cell 1.'
    )

context    = df_context_ilr.copy()
all_rows   = []
steps_done = 0

while steps_done < n_future_steps:
    window_len   = min(PREDICTION_LENGTH, n_future_steps - steps_done)
    current_last = int(context['time_idx'].max())
    ctx_slice    = context[context['time_idx'] >= current_last - ENCODER_LENGTH + 1].copy()

    # Append placeholder decoder rows for the prediction window. `from_dataset(predict=True)`
    # needs encoder + decoder rows to slice out a prediction. The 'value' column is masked
    # in the decoder anyway, but `time_idx` and `quarter` (known covariates) MUST be real.
    placeholder_rows = []
    for series_id in ctx_slice['series'].unique():
        bgy, coord = split_series_id(series_id)
        for step in range(PREDICTION_LENGTH):
            ft = current_last + 1 + step
            placeholder_rows.append(dict(
                baranggay_id=bgy,
                time_idx=ft,
                year=ft // 4 + YEAR_ORIGIN,
                quarter=str(ft % 4 + 1),
                ilr_coord=coord,
                series=series_id,
                value=0.0,           # masked by predict=True
            ))
    ctx_slice = pd.concat([ctx_slice, pd.DataFrame(placeholder_rows)], ignore_index=True) \
                  .sort_values(['series', 'time_idx']).reset_index(drop=True)

    try:
        # Inherit encoders, feature ordering, and target normalizer from `training`
        # so the model sees exactly the input shape it was trained with.
        roll_ds = TimeSeriesDataSet.from_dataset(
            training, ctx_slice, predict=True, stop_randomization=True,
        )
    except Exception as e:
        print(f'[ERROR] step {steps_done}: {e}'); break

    if len(roll_ds) == 0:
        print(f'[ERROR] step {steps_done}: empty dataset.'); break

    print(f'  step {steps_done+1}/{n_future_steps}  '
          f'ctx [{ctx_slice.time_idx.min()}-{current_last}]  '
          f'predict [{current_last+1}-{current_last+window_len}]  '
          f'({len(roll_ds)} samples)')

    roll_dl = roll_ds.to_dataloader(
        train=False, batch_size=EVAL_BATCH_SIZE, num_workers=0, batch_sampler='synchronized',
    )
    with torch.no_grad():
        preds = best_model.predict(
            roll_dl, mode='samples', n_samples=N_SAMPLES,
            trainer_kwargs=dict(accelerator='cpu'), return_x=True,
        )

    si_ilr = list(roll_ds.x_to_index(preds.x)['series'])
    raw    = preds.output.numpy().copy()
    del preds; free()

    samp_ilr = raw.transpose(0, 2, 1) if raw.shape[2] == N_SAMPLES else raw
    del raw; free()

    # Invert ILR → simplex (with safety projection).
    samp_simplex, si_lc = ilr_samples_to_simplex(
        samp_ilr, si_ilr, PREDICTION_LENGTH, apply_projection=True,
    )
    mean_simplex = samp_simplex.mean(axis=1).astype(np.float32)

    # Mean of simplex samples is itself a valid simplex point (componentwise mean of
    # rows summing to 1 sums to 1), so we can use it as the next-step ground-truth
    # surrogate.  Convert that mean back to ILR to extend the rolling context.
    new_ctx_rows = []
    for step in range(window_len):
        ft = current_last + 1 + step
        fy = ft // 4 + YEAR_ORIGIN
        fq = ft %  4 + 1

        # Reassemble the predicted simplex composition per-barangay for this step.
        bgy_to_comp = {}
        for i, sid_lc in enumerate(si_lc):
            bgy, lc = _parse_bgy_lc_inline(sid_lc)
            smp_step = samp_simplex[i, :, step]
            all_rows.append(dict(
                baranggay_id=bgy, year=fy, quarter=fq, time_idx=ft, land_cover_type=lc,
                pred_mean=float(mean_simplex[i, step]),
                pred_lo50=float(np.percentile(smp_step, 25)),
                pred_hi50=float(np.percentile(smp_step, 75)),
                pred_lo90=float(np.percentile(smp_step,  5)),
                pred_hi90=float(np.percentile(smp_step, 95)),
            ))
            bgy_to_comp.setdefault(bgy, np.zeros(N_LC))
            bgy_to_comp[bgy][LAND_COVER_COLS.index(lc)] = float(mean_simplex[i, step])

        # Convert the recomposed simplex points back to ILR for context extension.
        for bgy, comp in bgy_to_comp.items():
            comp = np.clip(comp, 0.0, 1.0)
            comp = comp / max(comp.sum(), 1e-8)
            z = ilr_transform(comp[None, :])[0]
            for k, coord in enumerate(ILR_COORD_NAMES):
                new_ctx_rows.append(dict(
                    baranggay_id=bgy, time_idx=ft, year=fy, quarter=str(fq),
                    ilr_coord=coord, series=f'{bgy}_{coord}', value=float(z[k]),
                ))

    del samp_ilr, samp_simplex; free()
    context = pd.concat([context, pd.DataFrame(new_ctx_rows)], ignore_index=True) \
                .sort_values(['series', 'time_idx']).reset_index(drop=True)
    steps_done += window_len
    done_year = (current_last + window_len) // 4 + YEAR_ORIGIN
    done_qtr  = (current_last + window_len) %  4 + 1
    print(f'  -> rolled to {done_year} Q{done_qtr}  ({steps_done}/{n_future_steps})')

del context; free()
forecast_df = pd.DataFrame(all_rows)

if forecast_df.empty:
    print('[FAILED] forecast_df is empty — see [ERROR] messages above.')
else:
    print(f'\nForecast complete — {len(forecast_df)} rows | '
          f'{forecast_df["baranggay_id"].nunique()} barangays | '
          f'{forecast_df["land_cover_type"].nunique()} land cover types')
    print(forecast_df[['baranggay_id','year','quarter','land_cover_type','pred_mean']]
          .head(14).to_string(index=False))


INFO: GPU available: True (cuda), used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytor

Last observed : 2025 Q4  (time_idx=39)
Target        : 2026 Q1  (time_idx=40)
Steps to roll : 1
  step 1/1  ctx [28-39]  predict [40-40]  (774 samples)
  -> rolled to 2026 Q1  (1/1)

Forecast complete — 903 rows | 129 barangays | 7 land cover types
baranggay_id  year  quarter land_cover_type  pred_mean
         100  2026        1     bare_ground   0.000101
         100  2026        1      built_area   0.999058
         100  2026        1           crops   0.000101
         100  2026        1           grass   0.000101
         100  2026        1 shrub_and_scrub   0.000359
         100  2026        1           trees   0.000170
         100  2026        1           water   0.000110
         101  2026        1     bare_ground   0.000100
         101  2026        1      built_area   0.999401
         101  2026        1           crops   0.000100
         101  2026        1           grass   0.000100
         101  2026        1 shrub_and_scrub   0.000100
         101  2026        1         

In [ ]:
# ============================================================
#  CELL 14 — FORECAST VISUALISATIONS & EXPORT
# ============================================================
if forecast_df.empty:
    print('No forecast data to plot.')
else:
    pivot = (
        forecast_df
        .groupby(['year', 'quarter', 'land_cover_type'])['pred_mean']
        .mean().unstack('land_cover_type').round(4)
    )
    pivot.index = [f'{y} Q{q}' for y, q in pivot.index]
    print('Mean predicted land-cover fraction across all barangays:')
    print(pivot.to_string())

    # Rebuild observed-history DataFrame from the raw CSV in long-LC format
    # (the rolling-forecast context above is in ILR space, not LC space).
    df_hist_raw = pd.read_csv(DATA_PATH)
    df_hist_raw['baranggay_id'] = df_hist_raw['baranggay_id'].astype(str)
    df_hist_raw['time_idx']     = (df_hist_raw['year'] - YEAR_ORIGIN) * 4 \
                                 + (df_hist_raw['quarter'] - 1)
    df_hist = df_hist_raw.melt(
        id_vars=['baranggay_id', 'time_idx', 'year', 'quarter'],
        value_vars=LAND_COVER_COLS,
        var_name='land_cover_type', value_name='value',
    ).sort_values(['baranggay_id', 'land_cover_type', 'time_idx']).reset_index(drop=True)
    del df_hist_raw

    show_bgy = list(forecast_df['baranggay_id'].unique()[:3])

    for bgy in show_bgy:
        fig, axes = plt.subplots(2, 4, figsize=(18, 8), squeeze=False)
        axes = axes.flatten()
        for ax_i, lc in enumerate(LAND_COVER_COLS):
            ax   = axes[ax_i]
            hist = df_hist[(df_hist['baranggay_id'] == bgy) &
                           (df_hist['land_cover_type'] == lc)].sort_values('time_idx')
            fc   = forecast_df[(forecast_df['baranggay_id'] == bgy) &
                               (forecast_df['land_cover_type'] == lc)].sort_values('time_idx')
            ax.plot(hist['time_idx'], hist['value'], color='steelblue', linewidth=1.5, label='Observed')
            if not fc.empty:
                t = fc['time_idx'].values
                ax.fill_between(t, fc['pred_lo90'], fc['pred_hi90'], alpha=0.18, color='tomato', label='90% PI')
                ax.fill_between(t, fc['pred_lo50'], fc['pred_hi50'], alpha=0.38, color='tomato', label='50% PI')
                ax.plot(t, fc['pred_mean'], color='tomato', linewidth=1.8, linestyle='--',
                        marker='o', markersize=4, label='Forecast')
                ax.axvline(t[0] - 0.5, color='gray', linestyle=':', linewidth=1)
            tick_t = [tt for tt in list(hist['time_idx']) + (list(fc['time_idx']) if not fc.empty else [])
                      if tt % 4 == 0]
            ax.set_xticks(tick_t)
            ax.set_xticklabels([str(tt // 4 + YEAR_ORIGIN) for tt in tick_t], rotation=45, fontsize=6)
            ax.set_title(lc, fontsize=9); ax.set_ylabel('Fraction', fontsize=7)
            ax.set_ylim(-0.02, 1.02); ax.tick_params(labelsize=7)
            if ax_i == 0: ax.legend(fontsize=7, loc='upper left')
        for j in range(len(LAND_COVER_COLS), len(axes)): axes[j].set_visible(False)
        fig.suptitle(f'Barangay {bgy} — Forecast to {TARGET_YEAR} Q{TARGET_QUARTER}', fontsize=13, y=1.01)
        plt.tight_layout(); plt.show(); plt.close(); free()

    del df_hist; free()

    out_path = f'deepvar_forecast_{TARGET_YEAR}_Q{TARGET_QUARTER}.csv'
    forecast_df.to_csv(out_path, index=False)
    print(f'\nSaved forecast → {out_path}  ({len(forecast_df)} rows)')
    print(f'Columns: {list(forecast_df.columns)}')


Mean predicted land-cover fraction across all barangays:
land_cover_type  bare_ground  built_area   crops   grass  shrub_and_scrub   trees   water
2026 Q1               0.0006      0.8643  0.0004  0.0055           0.0017  0.1269  0.0005

Saved forecast → deepvar_forecast_2026_Q1.csv  (903 rows)
Columns: ['baranggay_id', 'year', 'quarter', 'time_idx', 'land_cover_type', 'pred_mean', 'pred_lo50', 'pred_hi50', 'pred_lo90', 'pred_hi90']


In [ ]:
# ============================================================
#  CELL 15 — PREDICT 10 TIMESTEPS FROM END OF TRAINING DATA
# ============================================================
# Same rolling-ILR mechanism as Cell 13 but anchored at the end of TRAINING data
# (time_idx ≤ training cutoff in the raw CSV) and rolled forward 10 steps.
N_FUTURE_STEPS_10 = 40

df_all_10 = pd.read_csv(DATA_PATH)
if YEAR_ORIGIN is None:
    YEAR_ORIGIN = int(df_all_10['year'].min())
df_all_10 = df_all_10[df_all_10['year'] <= TRAIN_UNTIL_YEAR].copy()
df_all_10 = _prep(df_all_10)
df_all_10_ilr = melt_to_ilr(df_all_10)
del df_all_10

context_10  = df_all_10_ilr.copy()
all_rows_10 = []
steps_done  = 0

while steps_done < N_FUTURE_STEPS_10:
    window_len   = min(PREDICTION_LENGTH, N_FUTURE_STEPS_10 - steps_done)
    current_last = int(context_10['time_idx'].max())
    ctx_slice    = context_10[context_10['time_idx'] >= current_last - ENCODER_LENGTH + 1].copy()

    # Append placeholder decoder rows (see Cell 13 for explanation).
    placeholder_rows = []
    for series_id in ctx_slice['series'].unique():
        bgy, coord = split_series_id(series_id)
        for step in range(PREDICTION_LENGTH):
            ft = current_last + 1 + step
            placeholder_rows.append(dict(
                baranggay_id=bgy, time_idx=ft,
                year=ft // 4 + YEAR_ORIGIN, quarter=str(ft % 4 + 1),
                ilr_coord=coord, series=series_id, value=0.0,
            ))
    ctx_slice = pd.concat([ctx_slice, pd.DataFrame(placeholder_rows)], ignore_index=True) \
                  .sort_values(['series', 'time_idx']).reset_index(drop=True)

    roll_ds = TimeSeriesDataSet.from_dataset(
        training, ctx_slice, predict=True, stop_randomization=True,
    )
    if len(roll_ds) == 0:
        print(f'[ERROR] step {steps_done}: empty dataset.'); break

    print(f'  step {steps_done+1}/{N_FUTURE_STEPS_10}  '
          f'predict [{current_last+1}-{current_last+window_len}]  '
          f'({len(roll_ds)} samples)')

    roll_dl = roll_ds.to_dataloader(
        train=False, batch_size=EVAL_BATCH_SIZE, num_workers=0, batch_sampler='synchronized',
    )
    with torch.no_grad():
        preds = best_model.predict(
            roll_dl, mode='samples', n_samples=N_SAMPLES,
            trainer_kwargs=dict(accelerator='cpu'), return_x=True,
        )

    si_ilr = list(roll_ds.x_to_index(preds.x)['series'])
    raw    = preds.output.numpy().copy()
    del preds; free()

    samp_ilr = raw.transpose(0, 2, 1) if raw.shape[2] == N_SAMPLES else raw
    del raw; free()
    samp_simplex, si_lc = ilr_samples_to_simplex(
        samp_ilr, si_ilr, PREDICTION_LENGTH, apply_projection=True,
    )
    mean_simplex = samp_simplex.mean(axis=1).astype(np.float32)

    new_ctx_rows = []
    for step in range(window_len):
        ft = current_last + 1 + step
        fy = ft // 4 + YEAR_ORIGIN
        fq = ft %  4 + 1
        bgy_to_comp = {}
        for i, sid_lc in enumerate(si_lc):
            bgy, lc = _parse_bgy_lc_inline(sid_lc)
            smp_step = samp_simplex[i, :, step]
            all_rows_10.append(dict(
                baranggay_id=bgy, year=fy, quarter=fq, time_idx=ft, land_cover_type=lc,
                pred_mean=float(mean_simplex[i, step]),
                pred_lo50=float(np.percentile(smp_step, 25)),
                pred_hi50=float(np.percentile(smp_step, 75)),
                pred_lo90=float(np.percentile(smp_step,  5)),
                pred_hi90=float(np.percentile(smp_step, 95)),
            ))
            bgy_to_comp.setdefault(bgy, np.zeros(N_LC))
            bgy_to_comp[bgy][LAND_COVER_COLS.index(lc)] = float(mean_simplex[i, step])

        for bgy, comp in bgy_to_comp.items():
            comp = np.clip(comp, 0.0, 1.0); comp = comp / max(comp.sum(), 1e-8)
            z = ilr_transform(comp[None, :])[0]
            for k, coord in enumerate(ILR_COORD_NAMES):
                new_ctx_rows.append(dict(
                    baranggay_id=bgy, time_idx=ft, year=fy, quarter=str(fq),
                    ilr_coord=coord, series=f'{bgy}_{coord}', value=float(z[k]),
                ))
    context_10 = pd.concat([context_10, pd.DataFrame(new_ctx_rows)], ignore_index=True) \
                   .sort_values(['series', 'time_idx']).reset_index(drop=True)
    steps_done += window_len

forecast_10_df = pd.DataFrame(all_rows_10)
out_path_10 = f'deepvar_forecast_10steps_from_{TRAIN_UNTIL_YEAR}.csv'
forecast_10_df.to_csv(out_path_10, index=False)
print(f'\n10-step forecast saved → {out_path_10}  ({len(forecast_10_df)} rows)')


INFO: GPU available: True (cuda), used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: False


  step 1/40  predict [32-35]  (774 samples)


INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litm

  step 5/40  predict [36-39]  (774 samples)


INFO: GPU available: True (cuda), used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytor

...

  step 29/40  predict [60-63]  (774 samples)


INFO: GPU available: True (cuda), used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytor

  step 33/40  predict [64-67]  (774 samples)


INFO: GPU available: True (cuda), used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytor

  step 37/40  predict [68-71]  (774 samples)

10-step forecast saved → deepvar_forecast_10steps_from_2023.csv  (36120 rows)
